# Module 6 — Randomized email experiment

Word guide steps 1.6.1–1.6.5. This notebook reads verified exports; the complete calculation is in `scripts/run_hillstrom.py`, `scripts/hillstrom_methods.py` and `sql/06_hillstrom.sql`. Monetary values are USD. This is a separate retailer experiment, not the grocery campaign population.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT=Path.cwd()
DATA=ROOT/'exports/hillstrom'
def load(name):
    return pd.read_csv(DATA/f'{name}.csv')
source=json.loads((ROOT/'docs/hillstrom_source.json').read_text(encoding='utf-8'))
print('Customers:', source['rows'], '| CSV SHA-256:', source['csv_sha256'])
display(load('hillstrom_arm_summary'))
display(load('hillstrom_srm'))
display(load('hillstrom_balance').sort_values('standardized_mean_difference',key=abs,ascending=False).head())

## 1. Compare outcomes per randomized customer

Keep zeros. Compare each version with no email; then compare the two versions directly. Rates use 0–1 units; 0.01 is one percentage point. All intervals are pointwise 95% bootstrap intervals, with 2,000 resamples and seed 42. Welch p values use Holm families of six primary and three direct contrasts; intervals are not multiplicity-adjusted.

In [ ]:
display(load('hillstrom_effects'))
display(load('email_head_to_head'))
display(Image(filename=str(ROOT/'charts/hillstrom_effects.png')))

## 2. Explore baseline customer cuts

Six cuts produce 126 exploratory comparisons with one Holm family. Merchandise purchase history does not identify customer sex or gender. A different significance label across two subgroups is not evidence of an interaction. Segment spending is sparse and noisy.

In [ ]:
segments=load('segment_effects')
display(segments[(segments['cut']=='buyer') & (segments['metric']=='visit')])
print('Exploratory tests:',len(segments),'| Cuts:',', '.join(segments['cut'].unique()))
display(Image(filename=str(ROOT/'charts/hillstrom_buyer_segments.png')))

## 3. Freeze targeting before looking at validation outcomes

Thirty observed cells out of 36 possible; the six both-lines/low-spend cells are empty. Random half seed 7 trains the better-version spend score per cell. Outcomes or assignments from the held-out half cannot choose rules. Whole cells select 8,486, so outcome-free seeded boundary ties fill the exact 10,000 quota. Suppression is a forced capacity ranking, not evidence of harmful email.

In [ ]:
rules=load('policy_rules')
display(load('policy_audience_counts'))
display(rules[(rules['seed']==7)&(rules['exact_target']>0)])
heldout=load('heldout_effects')
display(heldout[(heldout['scope']=='exact')&(heldout['metric']=='spend')&(heldout['arm']=='Mens E-Mail')])
display(Image(filename=str(ROOT/'charts/hillstrom_targeting_validation.png')))

## 4. Evaluate the policy, not just one arm in its audience

The learned per-cell version policy needs its own value estimate. The Horvitz–Thompson estimator uses the known 1/3 randomized assignment probability and a paired comparison against always sending Men's. Validation intervals condition on the trained rule. Seed 8 is an overlapping sensitivity split, not a new experiment: it fails to confirm a strong targeting gain. Do not tune the policy using these validation results.

In [ ]:
value=load('heldout_policy_value')
display(value[(value['metric']=='spend')&(value['audience']=='target')])
for decision in ['target','suppress']:
    candidates=load(decision+'_10000')
    assert len(candidates)==10000 and candidates['source_row'].is_unique
    print(decision, len(candidates), '| version suggestion status:',candidates['variant_suggestion_status'].unique())
display(load('hillstrom_profile'))
checks=load('hillstrom_checks')
assert checks['passed'].astype(str).str.lower().eq('true').all()
print(len(checks),'database checks passed')
display(checks)

## 5. Make a decision and learn the SQL workflow

Men's is supported as the population default; the selected cell version mix and ranking remain provisional. No email cost or margin is supplied, so do not claim net profit. Open `Open Hillstrom in DBeaver.cmd` and run `sql/06_hillstrom_learning.sql` one query at a time with Ctrl+Enter. The readout below answers all eight guide questions. Module 7 remains pending.

In [ ]:
display(Markdown((ROOT/'docs/experiment_readout.md').read_text(encoding='utf-8')))